# DM1 09E — Review-informed model and k selection
# DM1 09E — 复核驱动的模型与 k 选择

This notebook reuses **exactly the modeling population and text representations produced by 09c**. It does not inherit `k=12` as an answer. Instead, TF-IDF/LSA KMeans and weighted MiniLM KMeans are each evaluated independently for every `k=4–20`.

Selection uses the same five random seeds, the same 5,000-row evaluation subset, cluster-size diagnostics, term distinctiveness, and a common review pack. Baseline and advanced models are allowed to select different values of k. The primary model is chosen only after quantitative screening and a comparable interpretation review.

本 notebook 完整复用 09c 的建模样本和文本表示，但不把 `k=12` 当作预设答案。TF-IDF/LSA KMeans 与加权 MiniLM KMeans 分别在 `k=4–20` 上独立评价。两个方法可以选择不同的最佳 k。

In [1]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() or 1)

from pathlib import Path
import json
import math
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module=r"joblib\.externals\.loky\.backend\.context")

WORK_DIR = Path.cwd()
assignment_candidates = [
    WORK_DIR / "v2_controlled_cluster_assignments.parquet",
    WORK_DIR / "09c_outputs" / "v2_controlled_cluster_assignments.parquet",
    WORK_DIR.parent / "09c_outputs" / "v2_controlled_cluster_assignments.parquet",
    WORK_DIR / "v2_clustering_outputs_09c_review_informed" / "v2_controlled_cluster_assignments.parquet",
    WORK_DIR.parent / "v2_controlled_cluster_assignments.parquet",
    WORK_DIR.parent / "v2_clustering_outputs_09c_review_informed" / "v2_controlled_cluster_assignments.parquet",
]
ASSIGNMENT_PATH = next((path for path in assignment_candidates if path.exists()), assignment_candidates[0])
OUTPUT_DIR = WORK_DIR / "09e_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_VALUES = list(range(4, 21))
SEEDS = [7, 21, 42, 84, 168]
MAIN_SEED = 42
EVALUATION_N = 5_000
COMMENT_WEIGHT = 0.85
CONTEXT_WEIGHT = 0.15
EMBEDDING_EXPORT_PATH = OUTPUT_DIR / "weighted_minilm_embeddings.npy"

assert ASSIGNMENT_PATH.exists(), f"Input file not found: {ASSIGNMENT_PATH}"
print("Assignments:", ASSIGNMENT_PATH)
print("09E output:", OUTPUT_DIR)
print("k range:", K_VALUES[0], "to", K_VALUES[-1])
print("seeds:", SEEDS)

Assignments: D:\DM1_project\going to submit\09c_outputs\v2_controlled_cluster_assignments.parquet
09E output: D:\DM1_project\going to submit\09e_outputs
k range: 4 to 20
seeds: [7, 21, 42, 84, 168]


## 1. Reconstruct the exact 09c feature spaces / 重建 09c 的原始特征空间

The fitted 09c vectorizers and LSA transform are reused. The cached weighted MiniLM vectors were created from the same row order using 85% comment and 15% context weights. Assertions prevent silent sample or cache drift.

这里复用 09c 已拟合的向量器、LSA 变换和加权 MiniLM 缓存，并检查样本顺序与规模，避免悄悄改变数据。

In [2]:
df = pd.read_parquet(ASSIGNMENT_PATH)
assert len(df) == 28_230
assert df["comment_id"].is_unique
assert df.groupby("subreddit").size().nunique() == 1

comment_vectorizer = joblib.load(ASSIGNMENT_PATH.parent / "baseline_comment_tfidf_vectorizer.joblib")
context_vectorizer = joblib.load(ASSIGNMENT_PATH.parent / "baseline_context_tfidf_vectorizer.joblib")
lsa = joblib.load(ASSIGNMENT_PATH.parent / "baseline_lsa.joblib")
lsa_normalizer = joblib.load(ASSIGNMENT_PATH.parent / "baseline_lsa_normalizer.joblib")

X_comment = comment_vectorizer.transform(df["comment_representation_text"])
X_context = context_vectorizer.transform(df["topic_context_text"])
X_tfidf = sparse.hstack([
    X_comment * COMMENT_WEIGHT,
    X_context * CONTEXT_WEIGHT,
], format="csr", dtype=np.float32)
X_tfidf = normalize(X_tfidf, norm="l2", copy=False)
X_lsa = lsa_normalizer.transform(lsa.transform(X_tfidf)).astype(np.float32)

embedding_path = ASSIGNMENT_PATH.parent / "weighted_minilm_embeddings.npy"
assert embedding_path.exists(), embedding_path
X_semantic = np.load(embedding_path).astype(np.float32)
assert X_semantic.shape == (len(df), 384)
np.save(EMBEDDING_EXPORT_PATH, X_semantic)

evaluation_indices = np.random.default_rng(MAIN_SEED).choice(
    len(df), size=min(EVALUATION_N, len(df)), replace=False
)

print("Rows:", f"{len(df):,}")
print("LSA shape:", X_lsa.shape)
print("MiniLM shape:", X_semantic.shape)
print("Embedding source:", embedding_path)

Rows: 28,230
LSA shape: (28230, 100)
MiniLM shape: (28230, 384)
Embedding source: D:\DM1_project\going to submit\09c_outputs\weighted_minilm_embeddings.npy


## 2. Evaluate every k with identical controls / 使用完全一致的控制评价所有 k

For each method and k, the notebook records:

- cosine silhouette in both LSA and MiniLM spaces;
- pairwise ARI across five random seeds;
- minimum and maximum cluster shares;
- normalized cluster-size entropy; and
- the mean margin between the assigned and second-closest centroid in the method's native space.

No k is selected in this step. / 此步骤不选择任何 k。

In [3]:
def normalized_centroid_membership(X, labels, centers):
    centers = centers / np.maximum(np.linalg.norm(centers, axis=1, keepdims=True), 1e-12)
    sims = X @ centers.T
    own = sims[np.arange(len(labels)), labels]
    second = np.partition(sims, -2, axis=1)[:, -2]
    return own.astype(np.float32), (own - second).astype(np.float32)


def size_entropy(shares):
    values = shares.to_numpy(dtype=float)
    entropy = -(values * np.log(np.maximum(values, 1e-12))).sum()
    return float(entropy / np.log(len(values)))


spaces = {
    "tfidf_lsa_kmeans": X_lsa,
    "minilm_kmeans": X_semantic,
}
all_models = {}
all_labels = {}
all_membership = {}
metric_rows = []
stability_rows = []

for method, X_native in spaces.items():
    for k in K_VALUES:
        started = time.perf_counter()
        labels_by_seed = {}
        models_by_seed = {}
        for seed in SEEDS:
            model = KMeans(
                n_clusters=k, init="k-means++", n_init=10,
                max_iter=300, random_state=seed,
            )
            labels_by_seed[seed] = model.fit_predict(X_native)
            models_by_seed[seed] = model

        main_model = models_by_seed[MAIN_SEED]
        labels = labels_by_seed[MAIN_SEED]
        shares = pd.Series(labels).value_counts(normalize=True).sort_index()
        membership, margin = normalized_centroid_membership(X_native, labels, main_model.cluster_centers_)

        pair_aris = []
        for i, seed_a in enumerate(SEEDS):
            for seed_b in SEEDS[i + 1:]:
                ari = adjusted_rand_score(labels_by_seed[seed_a], labels_by_seed[seed_b])
                pair_aris.append(ari)
                stability_rows.append({
                    "method": method, "k": k,
                    "seed_a": seed_a, "seed_b": seed_b, "ARI": ari,
                })

        eval_labels = labels[evaluation_indices]
        sil_lsa = silhouette_score(
            X_lsa[evaluation_indices], eval_labels, metric="cosine"
        )
        sil_semantic = silhouette_score(
            X_semantic[evaluation_indices], eval_labels, metric="cosine"
        )
        native_sil = sil_lsa if method == "tfidf_lsa_kmeans" else sil_semantic
        other_sil = sil_semantic if method == "tfidf_lsa_kmeans" else sil_lsa

        metric_rows.append({
            "method": method,
            "k": k,
            "silhouette_lsa": sil_lsa,
            "silhouette_minilm": sil_semantic,
            "silhouette_native": native_sil,
            "silhouette_other_space": other_sil,
            "mean_seed_ari": float(np.mean(pair_aris)),
            "minimum_seed_ari": float(np.min(pair_aris)),
            "minimum_cluster_share": float(shares.min()),
            "maximum_cluster_share": float(shares.max()),
            "normalized_size_entropy": size_entropy(shares),
            "mean_native_membership": float(membership.mean()),
            "mean_native_margin": float(margin.mean()),
            "seconds": time.perf_counter() - started,
        })
        all_models[(method, k)] = main_model
        all_labels[(method, k)] = labels.astype(np.int16)
        all_membership[(method, k)] = membership
        print(
            method, f"k={k}",
            f"native_sil={native_sil:.4f}",
            f"ARI={np.mean(pair_aris):.3f}",
            f"share={shares.min():.3f}-{shares.max():.3f}",
        )

metrics = pd.DataFrame(metric_rows)
stability_pairs = pd.DataFrame(stability_rows)
metrics

tfidf_lsa_kmeans k=4 native_sil=0.0645 ARI=0.823 share=0.080-0.395


tfidf_lsa_kmeans k=5 native_sil=0.0715 ARI=0.997 share=0.077-0.341


tfidf_lsa_kmeans k=6 native_sil=0.0775 ARI=0.700 share=0.065-0.303


tfidf_lsa_kmeans k=7 native_sil=0.0807 ARI=0.689 share=0.051-0.315


tfidf_lsa_kmeans k=8 native_sil=0.0866 ARI=0.681 share=0.060-0.253


...

minilm_kmeans k=17 native_sil=0.0475 ARI=0.910 share=0.027-0.087


minilm_kmeans k=18 native_sil=0.0498 ARI=0.769 share=0.028-0.089


minilm_kmeans k=19 native_sil=0.0480 ARI=0.795 share=0.026-0.087


minilm_kmeans k=20 native_sil=0.0481 ARI=0.903 share=0.022-0.087


,method,k,silhouette_lsa,silhouette_minilm,silhouette_native,silhouette_other_space,mean_seed_ari,minimum_seed_ari,minimum_cluster_share,maximum_cluster_share,normalized_size_entropy,mean_native_membership,mean_native_margin,seconds
0,tfidf_lsa_kmeans,4,0.064490,-0.017222,0.064490,-0.017222,0.822840,0.665833,0.079844,0.395289,0.902001,0.415158,0.139180,2.584237
1,tfidf_lsa_kmeans,5,0.071463,-0.009210,0.071463,-0.009210,0.996868,0.993304,0.076727,0.341410,0.942852,0.430127,0.137717,2.520093
2,tfidf_lsa_kmeans,6,0.077550,-0.006124,0.077550,-0.006124,0.700092,0.568957,0.065143,0.302976,0.932166,0.442741,0.143614,2.650777
3,tfidf_lsa_kmeans,7,0.080718,-0.023873,0.080718,-0.023873,0.689022,0.546585,0.050549,0.314984,0.908440,0.453311,0.149928,2.809444
4,tfidf_lsa_kmeans,8,0.086603,0.010256,0.086603,0.010256,0.680961,0.507961,0.060468,0.252674,0.932269,0.464343,0.153294,3.196737
5,tfidf_lsa_kmeans,9,0.088354,-0.003308,0.088354,-0.003308,0.653425,0.515486,0.042650,0.196635,0.944930,0.474237,0.151766,3.327753
6,tfidf_lsa_kmeans,10,0.095652,-0.004437,0.095652,-0.004437,0.672713,0.547068,0.044810,0.253170,0.933877,0.483194,0.157733,3.695520
7,tfidf_lsa_kmeans,11,0.098418,0.000945,0.098418,0.000945,0.801473,0.654599,0.041162,0.202869,0.949324,0.491257,0.156662,4.265333
8,tfidf_lsa_kmeans,12,0.100916,0.001804,0.100916,0.001804,0.816102,0.727123,0.030677,0.202196,0.943078,0.497436,0.159518,4.799144
9,tfidf_lsa_kmeans,13,0.104403,0.000142,0.104403,0.000142,0.846342,0.767841,0.031243,0.180198,0.948394,0.504902,0.159213,5.076187


## 3. Add representation-independent topic-term diagnostics / 添加跨表示的主题词诊断

All assignments are described from the same cleaned comment text. Class-based TF-IDF identifies the top terms for every cluster. Two diagnostics are added:

- **topic diversity**: the share of unique terms among all top-10 topic terms;
- **term exclusivity**: how strongly a selected term belongs to one cluster rather than appearing throughout the corpus.

These are interpretation diagnostics, not substitutes for reading comments. / 它们是解释性诊断，不能替代评论阅读。

In [4]:
domain_stop = {
    "climate", "change", "global", "people", "person", "just", "like", "don",
    "doesn", "didn", "isn", "aren", "think", "really", "thing", "things",
    "way", "know", "make", "makes", "want", "need", "time", "times", "year",
    "years", "world", "say", "says", "said", "ve", "re", "ll", "im", "reddit",
    "https", "http", "com", "amp", "would", "could", "also", "much", "many",
    "get", "got", "going", "one", "even",
}
term_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=sorted(set(ENGLISH_STOP_WORDS).union(domain_stop)),
    ngram_range=(1, 2), min_df=10, max_df=0.85,
    max_features=40_000,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9\-]{1,}\b",
)
X_terms = term_vectorizer.fit_transform(df["topic_comment_text"].fillna(""))
term_names = np.asarray(term_vectorizer.get_feature_names_out())
global_counts = np.asarray(X_terms.sum(axis=0)).ravel()


def topic_term_diagnostics(labels, k, top_n=10):
    membership_matrix = sparse.csr_matrix(
        (np.ones(len(labels)), (labels, np.arange(len(labels)))),
        shape=(k, len(labels)),
    )
    cluster_counts = (membership_matrix @ X_terms).tocsr()
    class_lengths = np.asarray(cluster_counts.sum(axis=1)).ravel()
    idf = np.log1p(class_lengths.mean() / np.maximum(global_counts, 1))
    rows = []
    chosen_all = []
    exclusivities = []
    for cluster in range(k):
        counts = cluster_counts.getrow(cluster).toarray().ravel()
        scores = (counts / max(class_lengths[cluster], 1)) * idf
        order = np.argsort(scores)[::-1]
        chosen = []
        for idx in order:
            term = term_names[idx]
            if any(term == old or term in old or old in term for old in chosen):
                continue
            chosen.append(term)
            exclusivity = float(counts[idx] / max(global_counts[idx], 1))
            exclusivities.append(exclusivity)
            rows.append({
                "cluster": cluster,
                "rank": len(chosen),
                "term": term,
                "ctfidf_score": float(scores[idx]),
                "term_exclusivity": exclusivity,
            })
            if len(chosen) == top_n:
                break
        chosen_all.extend(chosen)
    diversity = len(set(chosen_all)) / max(len(chosen_all), 1)
    return pd.DataFrame(rows), float(diversity), float(np.mean(exclusivities))


term_frames = []
term_metric_rows = []
for (method, k), labels in all_labels.items():
    term_df, diversity, exclusivity = topic_term_diagnostics(labels, k)
    term_df.insert(0, "k", k)
    term_df.insert(0, "method", method)
    term_frames.append(term_df)
    term_metric_rows.append({
        "method": method, "k": k,
        "topic_diversity": diversity,
        "mean_term_exclusivity": exclusivity,
    })

all_topic_terms = pd.concat(term_frames, ignore_index=True)
metrics = metrics.merge(pd.DataFrame(term_metric_rows), on=["method", "k"], validate="one_to_one")
metrics

,method,k,silhouette_lsa,silhouette_minilm,silhouette_native,silhouette_other_space,mean_seed_ari,minimum_seed_ari,minimum_cluster_share,maximum_cluster_share,normalized_size_entropy,mean_native_membership,mean_native_margin,seconds,topic_diversity,mean_term_exclusivity
0,tfidf_lsa_kmeans,4,0.064490,-0.017222,0.064490,-0.017222,0.822840,0.665833,0.079844,0.395289,0.902001,0.415158,0.139180,2.584237,0.875000,0.497032
1,tfidf_lsa_kmeans,5,0.071463,-0.009210,0.071463,-0.009210,0.996868,0.993304,0.076727,0.341410,0.942852,0.430127,0.137717,2.520093,0.880000,0.480683
2,tfidf_lsa_kmeans,6,0.077550,-0.006124,0.077550,-0.006124,0.700092,0.568957,0.065143,0.302976,0.932166,0.442741,0.143614,2.650777,0.850000,0.424811
3,tfidf_lsa_kmeans,7,0.080718,-0.023873,0.080718,-0.023873,0.689022,0.546585,0.050549,0.314984,0.908440,0.453311,0.149928,2.809444,0.828571,0.371654
4,tfidf_lsa_kmeans,8,0.086603,0.010256,0.086603,0.010256,0.680961,0.507961,0.060468,0.252674,0.932269,0.464343,0.153294,3.196737,0.787500,0.389952
5,tfidf_lsa_kmeans,9,0.088354,-0.003308,0.088354,-0.003308,0.653425,0.515486,0.042650,0.196635,0.944930,0.474237,0.151766,3.327753,0.788889,0.359363
6,tfidf_lsa_kmeans,10,0.095652,-0.004437,0.095652,-0.004437,0.672713,0.547068,0.044810,0.253170,0.933877,0.483194,0.157733,3.695520,0.730000,0.333950
7,tfidf_lsa_kmeans,11,0.098418,0.000945,0.098418,0.000945,0.801473,0.654599,0.041162,0.202869,0.949324,0.491257,0.156662,4.265333,0.736364,0.324427
8,tfidf_lsa_kmeans,12,0.100916,0.001804,0.100916,0.001804,0.816102,0.727123,0.030677,0.202196,0.943078,0.497436,0.159518,4.799144,0.733333,0.317757
9,tfidf_lsa_kmeans,13,0.104403,0.000142,0.104403,0.000142,0.846342,0.767841,0.031243,0.180198,0.948394,0.504902,0.159213,5.076187,0.700000,0.302898


## 4. Screening and shortlist / 筛选与候选名单

Before ranking, solutions are required to have no cluster smaller than 2%, no cluster larger than 40%, and mean seed ARI of at least 0.70. Within each method, seven diagnostics are min-max scaled and combined with declared weights. The **top four passing solutions from each method** enter the common interpretation review. Four are retained—not three—so the primary semantic representation is not decided from an overly narrow score-only screen. The score creates the candidate set; it is not treated as an automatic substantive verdict.

在排序前，方案必须满足：最小簇不低于 2%、最大簇不高于 40%、平均种子 ARI 不低于 0.70。每种方法中，通过门槛且量化综合分最高的 **四个方案**进入统一解释性复核。候选数设为四而不是三，是为了避免主要语义表示被过窄的纯分数筛选提前决定。综合分只用于形成候选集合，不能自动替代语义复核。


In [5]:
SELECTION_WEIGHTS = {
    "mean_seed_ari": 0.25,
    "silhouette_native": 0.20,
    "mean_term_exclusivity": 0.15,
    "topic_diversity": 0.10,
    "normalized_size_entropy": 0.10,
    "minimum_cluster_share": 0.10,
    "mean_native_margin": 0.10,
}

metrics["passes_guardrails"] = (
    metrics["minimum_cluster_share"].ge(0.02)
    & metrics["maximum_cluster_share"].le(0.40)
    & metrics["mean_seed_ari"].ge(0.70)
)


def minmax(series):
    lo, hi = series.min(), series.max()
    if math.isclose(float(lo), float(hi)):
        return pd.Series(1.0, index=series.index)
    return (series - lo) / (hi - lo)


scored_parts = []
for method, group in metrics.groupby("method", sort=False):
    group = group.copy()
    eligible = group["passes_guardrails"]
    for column in SELECTION_WEIGHTS:
        group[column + "_scaled"] = minmax(group[column])
    group["quantitative_score"] = sum(
        weight * group[column + "_scaled"]
        for column, weight in SELECTION_WEIGHTS.items()
    )
    group["quantitative_rank"] = group["quantitative_score"].rank(
        method="min", ascending=False
    ).astype(int)
    scored_parts.append(group)

selection_scores = pd.concat(scored_parts, ignore_index=True)
shortlist = (
    selection_scores.loc[selection_scores["passes_guardrails"]]
    .sort_values(["method", "quantitative_score"], ascending=[True, False])
    .groupby("method", as_index=False, group_keys=False)
    .head(4)
    .copy()
)

display_columns = [
    "method", "k", "passes_guardrails", "quantitative_rank", "quantitative_score",
    "silhouette_lsa", "silhouette_minilm", "mean_seed_ari",
    "minimum_cluster_share", "maximum_cluster_share", "normalized_size_entropy",
    "topic_diversity", "mean_term_exclusivity", "mean_native_margin",
]
display(selection_scores[display_columns].sort_values(["method", "quantitative_rank"]))
print("Shortlist / 候选名单")
display(shortlist[display_columns])

,method,k,passes_guardrails,quantitative_rank,quantitative_score,silhouette_lsa,silhouette_minilm,mean_seed_ari,minimum_cluster_share,maximum_cluster_share,normalized_size_entropy,topic_diversity,mean_term_exclusivity,mean_native_margin
19,minilm_kmeans,6,True,1,0.786647,0.038471,0.068002,0.994442,0.067552,0.236132,0.969295,0.883333,0.474746,0.083036
20,minilm_kmeans,7,True,2,0.746159,0.034356,0.064836,0.987070,0.066844,0.188169,0.982086,0.857143,0.440551,0.082417
17,minilm_kmeans,4,True,3,0.740773,0.041604,0.082186,0.827205,0.070067,0.354410,0.909442,0.900000,0.590811,0.094045
22,minilm_kmeans,9,True,4,0.669916,0.032322,0.070545,0.975249,0.048176,0.155898,0.974821,0.788889,0.396752,0.082854
21,minilm_kmeans,8,True,5,0.654433,0.036934,0.066794,0.973893,0.047680,0.177471,0.969404,0.800000,0.416407,0.083149
18,minilm_kmeans,5,True,6,0.608766,0.039640,0.077196,0.731102,0.068650,0.306518,0.948402,0.860000,0.513169,0.092761
24,minilm_kmeans,11,True,7,0.525736,0.032292,0.056533,0.995782,0.040914,0.134998,0.975983,0.727273,0.339634,0.078774
25,minilm_kmeans,12,True,8,0.505306,0.030650,0.057099,0.975297,0.040666,0.126461,0.979363,0.733333,0.334578,0.076385
26,minilm_kmeans,13,True,9,0.413173,0.030238,0.057717,0.906057,0.029508,0.125292,0.972887,0.730769,0.331515,0.077477
23,minilm_kmeans,10,True,10,0.394739,0.031088,0.057007,0.813166,0.047821,0.137053,0.981334,0.760000,0.360286,0.076809


Shortlist / 候选名单


,method,k,passes_guardrails,quantitative_rank,quantitative_score,silhouette_lsa,silhouette_minilm,mean_seed_ari,minimum_cluster_share,maximum_cluster_share,normalized_size_entropy,topic_diversity,mean_term_exclusivity,mean_native_margin
19,minilm_kmeans,6,True,1,0.786647,0.038471,0.068002,0.994442,0.067552,0.236132,0.969295,0.883333,0.474746,0.083036
20,minilm_kmeans,7,True,2,0.746159,0.034356,0.064836,0.987070,0.066844,0.188169,0.982086,0.857143,0.440551,0.082417
17,minilm_kmeans,4,True,3,0.740773,0.041604,0.082186,0.827205,0.070067,0.354410,0.909442,0.900000,0.590811,0.094045
22,minilm_kmeans,9,True,4,0.669916,0.032322,0.070545,0.975249,0.048176,0.155898,0.974821,0.788889,0.396752,0.082854
1,tfidf_lsa_kmeans,5,True,1,0.686655,0.071463,-0.009210,0.996868,0.076727,0.341410,0.942852,0.880000,0.480683,0.137717
9,tfidf_lsa_kmeans,13,True,2,0.534263,0.104403,0.000142,0.846342,0.031243,0.180198,0.948394,0.700000,0.302898,0.159213
7,tfidf_lsa_kmeans,11,True,3,0.514802,0.098418,0.000945,0.801473,0.041162,0.202869,0.949324,0.736364,0.324427,0.156662
8,tfidf_lsa_kmeans,12,True,4,0.511478,0.100916,0.001804,0.816102,0.030677,0.202196,0.943078,0.733333,0.317757,0.159518


## 5. Create one comparable interpretation pack / 生成统一解释性复核包

For every shortlisted solution, each cluster contributes three centroid-nearest comments and three random comments from distinct posts. The same sampling rule is used for both methods and every k. This pack supports a blinded comparison of clarity, overlap, and residual structure.

每个候选方案的每个簇都抽取 3 条中心评论和 3 条不同帖子的随机评论，所有方法和 k 使用同一规则。

In [6]:
review_rows = []
profile_rows = []

for candidate in shortlist.itertuples(index=False):
    method, k = candidate.method, int(candidate.k)
    labels = all_labels[(method, k)]
    strength = all_membership[(method, k)]
    terms = all_topic_terms.loc[
        all_topic_terms["method"].eq(method) & all_topic_terms["k"].eq(k)
    ]
    term_map = terms.groupby("cluster")["term"].apply(lambda x: ", ".join(x)).to_dict()

    candidate_df = df[[
        "comment_id", "post_id", "subreddit", "post_title", "clean_text",
        "comment_representation_text", "topic_context_text", "review_gate_score",
    ]].copy()
    candidate_df["cluster"] = labels
    candidate_df["membership_strength"] = strength

    for cluster, group in candidate_df.groupby("cluster"):
        unique_posts = group.sort_values("membership_strength", ascending=False).drop_duplicates("post_id")
        centroid = unique_posts.head(3)
        remaining = unique_posts.drop(index=centroid.index)
        random = remaining.sample(n=min(3, len(remaining)), random_state=MAIN_SEED + int(cluster))
        profile_rows.append({
            "method": method, "k": k, "cluster": int(cluster),
            "size": int(len(group)),
            "share": float(len(group) / len(candidate_df)),
            "mean_membership": float(group["membership_strength"].mean()),
            "top_terms": term_map[int(cluster)],
        })
        for sample_type, chosen in [("centroid", centroid), ("random", random)]:
            for rank, row in enumerate(chosen.itertuples(index=False), start=1):
                review_rows.append({
                    "method": method, "k": k, "cluster": int(cluster),
                    "sample_type": sample_type, "rank": rank,
                    "comment_id": row.comment_id, "post_id": row.post_id,
                    "subreddit": row.subreddit, "post_title": row.post_title,
                    "clean_text": row.clean_text,
                    "comment_representation_text": row.comment_representation_text,
                    "topic_context_text": row.topic_context_text,
                    "review_gate_score": row.review_gate_score,
                    "membership_strength": row.membership_strength,
                    "top_terms": term_map[int(cluster)],
                })

candidate_profiles = pd.DataFrame(profile_rows)
candidate_review_pack = pd.DataFrame(review_rows)
display(candidate_profiles.head(30))
print("Review rows:", len(candidate_review_pack))

,method,k,cluster,size,share,mean_membership,top_terms
0,minilm_kmeans,6,0,5729,0.202940,0.626398,"solar, energy, power, nuclear, wind, grid, ren..."
1,minilm_kmeans,6,1,4809,0.170351,0.534302,"oil, energy, trump, fossil, gas, companies, mo..."
2,minilm_kmeans,6,2,6666,0.236132,0.650406,"warming, planet, problem, humans, real, life, ..."
3,minilm_kmeans,6,3,1907,0.067552,0.637232,"ev, car, electric, battery, vehicles, ice, bat..."
4,minilm_kmeans,6,4,4393,0.155615,0.583816,"warming, temperature, ice, weather, earth, dat..."
5,minilm_kmeans,6,5,4726,0.167411,0.616999,"carbon, co2, emissions, fossil, energy, use, a..."
6,minilm_kmeans,7,0,4075,0.144350,0.577011,"oil, energy, fossil, gas, trump, companies, fu..."
7,minilm_kmeans,7,1,5312,0.188169,0.626072,"solar, energy, power, nuclear, wind, grid, ele..."
8,minilm_kmeans,7,2,4007,0.141941,0.630763,"co2, carbon, emissions, atmosphere, fossil, me..."
9,minilm_kmeans,7,3,4051,0.143500,0.568469,"planet, life, human, food, population, water, ..."


Review rows: 402


## 6. Quantitative winners and exports / 量化结果与导出

The highest quantitative score in each method is recorded first. The final choice also uses the common interpretation sample described above.

每种方法先记录量化综合分最高的方案；最终选择还会结合上方统一口径的解释样本。

In [7]:
provisional_winners = (
    selection_scores.loc[selection_scores["passes_guardrails"]]
    .sort_values(["method", "quantitative_score"], ascending=[True, False])
    .groupby("method", as_index=False, group_keys=False)
    .head(1)
    .copy()
)

metrics.to_csv(OUTPUT_DIR / "all_k_raw_metrics.csv", index=False, encoding="utf-8-sig")
stability_pairs.to_csv(OUTPUT_DIR / "all_k_stability_pairs.csv", index=False, encoding="utf-8-sig")
selection_scores.to_csv(OUTPUT_DIR / "all_k_selection_scores.csv", index=False, encoding="utf-8-sig")
shortlist.to_csv(OUTPUT_DIR / "shortlisted_solutions.csv", index=False, encoding="utf-8-sig")
all_topic_terms.to_csv(OUTPUT_DIR / "all_k_topic_terms.csv", index=False, encoding="utf-8-sig")
candidate_profiles.to_csv(OUTPUT_DIR / "candidate_topic_profiles.csv", index=False, encoding="utf-8-sig")
candidate_review_pack.to_csv(OUTPUT_DIR / "candidate_interpretation_review_pack.csv", index=False, encoding="utf-8-sig")

np.savez_compressed(
    OUTPUT_DIR / "all_k_labels.npz",
    **{f"{method}__k{k}": labels for (method, k), labels in all_labels.items()},
)

for row in provisional_winners.itertuples(index=False):
    joblib.dump(
        all_models[(row.method, int(row.k))],
        OUTPUT_DIR / f"provisional_{row.method}_k{int(row.k)}.joblib",
    )

decision = {
    "status": "provisional_pending_comparable_interpretation_review",
    "data_source": str(ASSIGNMENT_PATH),
    "n_comments": int(len(df)),
    "k_range": [min(K_VALUES), max(K_VALUES)],
    "seeds": SEEDS,
    "guardrails": {
        "minimum_cluster_share": 0.02,
        "maximum_cluster_share": 0.40,
        "minimum_mean_seed_ari": 0.70,
    },
    "selection_weights": SELECTION_WEIGHTS,
    "provisional_winners": provisional_winners[
        ["method", "k", "quantitative_score", "quantitative_rank"]
    ].to_dict("records"),
    "shortlist": shortlist[["method", "k", "quantitative_score"]].to_dict("records"),
    "interpretation_review_required": True,
    "note": "No k is inherited from 09. Final selection requires equal-rule review of shortlisted solutions.",
}
(OUTPUT_DIR / "provisional_selection_decision.json").write_text(
    json.dumps(decision, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(json.dumps(decision, ensure_ascii=False, indent=2))
print("Saved to:", OUTPUT_DIR)

{
  "status": "provisional_pending_comparable_interpretation_review",
  "data_source": "D:\\DM1_project\\going to submit\\09c_outputs\\v2_controlled_cluster_assignments.parquet",
  "n_comments": 28230,
  "k_range": [
    4,
    20
  ],
  "seeds": [
    7,
    21,
    42,
    84,
    168
  ],
  "guardrails": {
    "minimum_cluster_share": 0.02,
    "maximum_cluster_share": 0.4,
    "minimum_mean_seed_ari": 0.7
  },
  "selection_weights": {
    "mean_seed_ari": 0.25,
    "silhouette_native": 0.2,
    "mean_term_exclusivity": 0.15,
    "topic_diversity": 0.1,
    "normalized_size_entropy": 0.1,
    "minimum_cluster_share": 0.1,
    "mean_native_margin": 0.1
  },
  "provisional_winners": [
    {
      "method": "minilm_kmeans",
      "k": 6,
      "quantitative_score": 0.786647012427008,
      "quantitative_rank": 1
    },
    {
      "method": "tfidf_lsa_kmeans",
      "k": 5,
      "quantitative_score": 0.6866545618132642,
      "quantitative_rank": 1
    }
  ],
  "shortlist": [
    {
  

## Interpretation rule / 解释规则

Do not call the provisional winner the final model until the shortlisted review pack has been examined with the same rubric. The final decision should record, for every shortlisted solution, the number of clear, broad, mixed, duplicate, and residual clusters and the fit of random comments. 09F must load the final—not provisional—decision.

在使用同一量表完成候选方案复核前，不得把暂定优胜方案称为最终模型。最终决定必须记录每个候选方案的清晰、宽泛、混合、重复和残余簇数量，以及随机评论的匹配情况。09F 只能读取最终决定。

## 7. Generate the full comparable interpretation review / 生成完整的统一解释性复核

The quantitative score narrows the search but does not by itself determine the final number of topics. The top-four rule above places MiniLM k=4, 6, 7, 9 and TF-IDF/LSA k=5, 11, 12, 13 into the candidate set. This section generates the same cluster profiles and the same centroid-plus-random reading sample for all eight candidates. Therefore, k=9 is visibly admitted by the declared shortlist rule rather than introduced after shortlisting.

量化综合分用于缩小候选范围，但不能单独决定最终主题数。上面的 Top-4 规则将 MiniLM k=4、6、7、9，以及 TF-IDF/LSA k=5、11、12、13 正式纳入候选集合。本节为全部八个候选生成相同口径的簇画像和“中心样本 + 随机样本”阅读材料。因此，k=9 是按照公开候选规则进入复核，而不是在候选筛选结束后才被加入。


In [8]:
extended_candidates = [
    (
        row.method,
        int(row.k),
        X_semantic if row.method == "minilm_kmeans" else X_lsa,
    )
    for row in shortlist.sort_values(["method", "quantitative_rank"]).itertuples(index=False)
]


def build_interpretation_pack(candidates):
    review_rows = []
    profile_rows = []
    for method, k, _ in candidates:
        labels = all_labels[(method, k)]
        strength = all_membership[(method, k)]
        terms = all_topic_terms.loc[
            all_topic_terms["method"].eq(method) & all_topic_terms["k"].eq(k)
        ]
        term_map = terms.groupby("cluster")["term"].apply(lambda x: ", ".join(x)).to_dict()

        candidate_df = df[[
            "comment_id", "post_id", "subreddit", "post_title", "clean_text",
            "comment_representation_text", "topic_context_text", "review_gate_score",
        ]].copy()
        candidate_df["cluster"] = labels
        candidate_df["membership_strength"] = strength

        for cluster, group in candidate_df.groupby("cluster"):
            unique_posts = (
                group.sort_values("membership_strength", ascending=False)
                .drop_duplicates("post_id")
            )
            centroid = unique_posts.head(3)
            remaining = unique_posts.drop(index=centroid.index)
            random = remaining.sample(
                n=min(3, len(remaining)), random_state=MAIN_SEED + int(cluster)
            )
            profile_rows.append({
                "method": method,
                "k": int(k),
                "cluster": int(cluster),
                "size": int(len(group)),
                "share": float(len(group) / len(candidate_df)),
                "mean_membership": float(group["membership_strength"].mean()),
                "top_terms": term_map[int(cluster)],
            })
            for sample_type, chosen in (("centroid", centroid), ("random", random)):
                for rank, row in enumerate(chosen.itertuples(index=False), start=1):
                    review_rows.append({
                        "method": method,
                        "k": int(k),
                        "cluster": int(cluster),
                        "sample_type": sample_type,
                        "rank": rank,
                        "comment_id": row.comment_id,
                        "post_id": row.post_id,
                        "subreddit": row.subreddit,
                        "post_title": row.post_title,
                        "clean_text": row.clean_text,
                        "comment_representation_text": row.comment_representation_text,
                        "topic_context_text": row.topic_context_text,
                        "review_gate_score": row.review_gate_score,
                        "membership_strength": row.membership_strength,
                        "top_terms": term_map[int(cluster)],
                    })
    return pd.DataFrame(profile_rows), pd.DataFrame(review_rows)


extended_candidate_profiles, extended_candidate_review_pack = build_interpretation_pack(
    extended_candidates
)
extended_candidate_profiles.to_csv(
    OUTPUT_DIR / "extended_candidate_topic_profiles.csv",
    index=False,
    encoding="utf-8-sig",
)
extended_candidate_review_pack.to_csv(
    OUTPUT_DIR / "extended_candidate_interpretation_review_pack.csv",
    index=False,
    encoding="utf-8-sig",
)

candidate_coverage = (
    extended_candidate_profiles.groupby(["method", "k"])
    .agg(clusters=("cluster", "nunique"), minimum_share=("share", "min"), maximum_share=("share", "max"))
    .reset_index()
)
display(candidate_coverage.round(4))
display(
    extended_candidate_profiles.loc[
        extended_candidate_profiles["method"].eq("minilm_kmeans")
        & extended_candidate_profiles["k"].eq(9)
    ].round(4)
)
print("Comparable review rows:", len(extended_candidate_review_pack))


,method,k,clusters,minimum_share,maximum_share
0,minilm_kmeans,4,4,0.0701,0.3544
1,minilm_kmeans,6,6,0.0676,0.2361
2,minilm_kmeans,7,7,0.0668,0.1882
3,minilm_kmeans,9,9,0.0482,0.1559
4,tfidf_lsa_kmeans,5,5,0.0767,0.3414
5,tfidf_lsa_kmeans,11,11,0.0412,0.2029
6,tfidf_lsa_kmeans,12,12,0.0307,0.2022
7,tfidf_lsa_kmeans,13,13,0.0312,0.1802


,method,k,cluster,size,share,mean_membership,top_terms
17,minilm_kmeans,9,0,3426,0.1214,0.6454,"planet, life, humans, earth, future, populatio..."
18,minilm_kmeans,9,1,1360,0.0482,0.6673,"nuclear, power, energy, fusion, reactors, plan..."
19,minilm_kmeans,9,2,4401,0.1559,0.6421,"science, real, warming, believe, scientists, r..."
20,minilm_kmeans,9,3,1824,0.0646,0.6401,"ev, car, electric, vehicles, battery, batterie..."
21,minilm_kmeans,9,4,3583,0.1269,0.5835,"temperature, warming, weather, ice, data, wate..."
22,minilm_kmeans,9,5,3760,0.1332,0.6244,"emissions, carbon, china, meat, energy, countr..."
23,minilm_kmeans,9,6,4027,0.1426,0.6352,"solar, energy, power, wind, grid, electricity,..."
24,minilm_kmeans,9,7,3561,0.1261,0.5869,"oil, energy, fossil, gas, trump, coal, fuels, ..."
25,minilm_kmeans,9,8,2288,0.0810,0.6383,"co2, carbon, atmosphere, methane, emissions, c..."


Comparable review rows: 402


## 8. Convert the interpretation review into auditable data / 将解释性复核转化为可审计数据

The review is recorded at the cluster level for **both representations** rather than entering a prewritten selected k. Every shortlisted MiniLM and TF-IDF/LSA cluster is classified as clear, broad, mixed, duplicate, or residual, with a short rationale. Size, share, membership strength, and distinctive terms are merged from the generated profiles, while the underlying centroid and random comments remain available in the review-pack CSV.

Candidate-level interpretability is then calculated with one declared formula:

`interpretability_score = (clear + 0.5 × broad − mixed − duplicate − residual) / k`

All candidates must first pass the quantitative guardrails. Within each representation, the code selects the highest interpretability score; the quantitative score is used only as a tie-breaker. Thus the final k is produced by the displayed cluster-level review data and calculation—not by a pre-filled `selected` label. These review judgments support model selection and are not treated as an independent gold standard.

两种表示的复核都以逐簇数据记录，而不是预先写入某个 selected k。进入候选名单的每个 MiniLM 与 TF-IDF/LSA 簇都会被标记为清晰、宽泛、混合、重复或残余，并附带简短理由；规模、占比、归属强度和区分性词语来自实际生成的画像，原始中心评论和随机评论则保存在复核包 CSV 中。

候选级可解释性使用同一公开公式计算：

`可解释性得分 =（清晰簇 + 0.5 × 宽泛簇 − 混合簇 − 重复簇 − 残余簇）/ k`

所有候选首先必须通过量化门槛。随后，每种表示选择可解释性得分最高者，量化综合分仅用于平局处理。因此最终 k 由逐簇复核数据和计算产生，而不是由预填的 `selected` 标签产生。这些复核结果用于模型选择，不作为独立金标准。


In [9]:
minilm_cluster_review_rows = [
    (4, 0, "clear", "Distinct EV, battery, and charging discussion."),
    (4, 1, "broad", "Combines temperature, weather, ice, data, and wider physical-climate evidence."),
    (4, 2, "broad", "Combines emissions, carbon, fossil fuels, and transition-economy discussion."),
    (4, 3, "mixed", "Nuclear, renewables, grids, coal, and system-cost discussion remain merged."),

    (6, 0, "mixed", "Nuclear and renewable-grid technologies remain in one energy-system cluster."),
    (6, 1, "clear", "Distinct fossil-fuel industry and transition political-economy discussion."),
    (6, 2, "broad", "Climate belief, future risk, and human-consequence discussion remain combined."),
    (6, 3, "clear", "Distinct EV, battery, and charging discussion."),
    (6, 4, "clear", "Distinct climate observation, weather, ice, and physical-impact discussion."),
    (6, 5, "broad", "Combines greenhouse gases, emissions responsibility, and carbon removal."),

    (7, 0, "clear", "Distinct fossil-fuel industry and transition political-economy discussion."),
    (7, 1, "broad", "Nuclear and renewable-grid technologies are still combined."),
    (7, 2, "broad", "Greenhouse gases, carbon emissions, and removal remain a broad carbon theme."),
    (7, 3, "broad", "Future risk, population, food, and human consequences form a broad theme."),
    (7, 4, "clear", "Distinct climate belief, denial, science-trust, and communication discussion."),
    (7, 5, "clear", "Distinct climate observation and physical-impact discussion."),
    (7, 6, "clear", "Distinct EV, battery, and charging discussion."),

    (9, 0, "broad", "Future climate risk, adaptation, population, and human consequences."),
    (9, 1, "clear", "Distinct nuclear power, fusion, reactor, cost, and waste discussion."),
    (9, 2, "broad", "Climate belief, denial, science trust, communication, and action attitudes."),
    (9, 3, "clear", "Distinct EV, battery, vehicle, and charging discussion."),
    (9, 4, "clear", "Distinct observations, temperature, weather, ice, oceans, and physical impacts."),
    (9, 5, "broad", "Emission responsibility, consumption, food, countries, and mitigation action."),
    (9, 6, "clear", "Distinct solar, wind, grid, storage, and renewable-system discussion."),
    (9, 7, "clear", "Distinct oil, gas, coal, industry, and transition political economy."),
    (9, 8, "clear", "Distinct greenhouse gases, carbon cycle, capture, sinks, and removal."),
]

minilm_cluster_review = pd.DataFrame(
    minilm_cluster_review_rows,
    columns=["k", "cluster", "review_status", "review_rationale"],
)
minilm_cluster_review.insert(0, "method", "minilm_kmeans")
minilm_cluster_review = minilm_cluster_review.merge(
    extended_candidate_profiles[[
        "method", "k", "cluster", "size", "share", "mean_membership", "top_terms"
    ]],
    on=["method", "k", "cluster"],
    how="left",
    validate="one_to_one",
)
assert minilm_cluster_review["top_terms"].notna().all()
def write_review_csv(frame, filename):
    target = OUTPUT_DIR / filename
    try:
        frame.to_csv(target, index=False, encoding="utf-8-sig")
        print("Saved:", target)
        return target
    except PermissionError:
        fallback = target.with_name(target.stem + "_latest" + target.suffix)
        frame.to_csv(fallback, index=False, encoding="utf-8-sig")
        print("Target is open in another program; saved instead:", fallback)
        return fallback


minilm_review_csv_path = write_review_csv(
    minilm_cluster_review,
    "minilm_candidate_cluster_review.csv",
)
display(minilm_cluster_review)

status_order = ["clear", "broad", "mixed", "duplicate", "residual"]
minilm_review_summary = (
    pd.crosstab(
        [minilm_cluster_review["method"], minilm_cluster_review["k"]],
        minilm_cluster_review["review_status"],
    )
    .reindex(columns=status_order, fill_value=0)
    .reset_index()
    .rename(columns={status: status + "_clusters" for status in status_order})
)
minilm_notes = pd.DataFrame([
    (4, "Too coarse: physical climate, carbon/fossil, and energy-system content remain broad or mixed."),
    (6, "Stable but too coarse: nuclear/renewables and belief/future-risk themes remain merged."),
    (7, "Clearer, but nuclear and renewable-grid discussion remains combined."),
    (9, "Nine substantive themes with no mixed, duplicate, or residual cluster in the structured review."),
], columns=["k", "review_note"])
minilm_review_summary = minilm_review_summary.merge(minilm_notes, on="k", validate="one_to_one")

baseline_cluster_review_rows = [
    (5, 0, "clear", "Coherent physical-climate observations and impacts: warming, temperature, ice, weather, oceans, and heat."),
    (5, 1, "broad", "Broad transition theme combining fossil fuels, renewables, industry, power, and carbon use."),
    (5, 2, "clear", "Coherent public-belief and science-trust discussion, although expressed in general language."),
    (5, 3, "broad", "Multiple electricity technologies remain combined: solar, nuclear, wind, grids, coal, and power plants."),
    (5, 4, "mixed", "Oil, emissions, EVs, China, corporate interests, and consumption remain in one large lexical cluster."),

    (11, 0, "broad", "Broad electricity-system theme combining nuclear, coal, renewables, grids, storage, and China."),
    (11, 1, "clear", "Distinct public belief, science trust, and perceived reality of climate change."),
    (11, 2, "broad", "Broad human-environment theme combining food, quality of life, the planet, and mitigation concerns."),
    (11, 3, "broad", "Physical-impact theme spans weather, temperature, ice, sea level, oceans, water, and heat."),
    (11, 4, "clear", "Distinct oil and gas industry, prices, production, companies, and political-economy discussion."),
    (11, 5, "clear", "Distinct solar, wind, panels, grids, batteries, electricity, and renewable-system costs."),
    (11, 6, "broad", "Broad greenhouse-gas and carbon-cycle theme covering CO2, atmosphere, methane, plants, water, and ppm."),
    (11, 7, "duplicate", "Fossil-fuel transition language overlaps the oil-and-gas and broad energy-system clusters."),
    (11, 8, "clear", "Distinct scientific attribution and consensus discussion about human-caused warming."),
    (11, 9, "clear", "Distinct electric-vehicle, battery, charging, and vehicle-technology discussion."),
    (11, 10, "clear", "Distinct national and per-capita emissions, China, population, and reduction discussion."),

    (12, 0, "broad", "Broad CO2, atmospheric concentration, methane, plants, temperature, and carbon-cycle discussion."),
    (12, 1, "clear", "Distinct weather, temperature, heat, cold, extreme-event, and observational-data discussion."),
    (12, 2, "clear", "Distinct solar, wind, panels, grids, batteries, electricity, and cost discussion."),
    (12, 3, "broad", "Broad energy-system theme combining nuclear, coal, renewables, grids, storage, and China."),
    (12, 4, "clear", "Distinct oil and gas industry, prices, production, companies, and political-economy discussion."),
    (12, 5, "clear", "Distinct scientific consensus and attribution discussion about human-caused warming."),
    (12, 6, "clear", "Distinct cryosphere and sea-level theme covering ice, oceans, melting, and sea-level rise."),
    (12, 7, "duplicate", "Carbon capture, footprint, tax, and reduction overlap the broader atmospheric-carbon theme."),
    (12, 8, "clear", "Distinct public-belief, science-trust, and perceived-reality discussion."),
    (12, 9, "duplicate", "Fossil-fuel and renewable-transition language overlaps the energy-system and oil-industry clusters."),
    (12, 10, "broad", "Broad environment, food, human welfare, emissions, and planetary-consequence discussion."),
    (12, 11, "clear", "Distinct electric-vehicle, battery, charging, and vehicle-technology discussion."),

    (13, 0, "broad", "Broad planetary, environmental, food, human-welfare, and general-problem discussion."),
    (13, 1, "broad", "Broad electricity-system theme combining coal, gas, renewables, grids, storage, and China."),
    (13, 2, "clear", "Distinct nuclear power, reactors, fusion, construction, cost, and waste discussion."),
    (13, 3, "clear", "Distinct national and per-capita emissions, China, population, and reduction discussion."),
    (13, 4, "clear", "Distinct weather, ice, temperature, sea-level, water, heat, and physical-impact discussion."),
    (13, 5, "clear", "Distinct atmospheric CO2, carbon capture, methane, trees, plants, and carbon-cycle discussion."),
    (13, 6, "clear", "Distinct electric-vehicle, battery, charging, and vehicle-technology discussion."),
    (13, 7, "clear", "Distinct public-belief, science-trust, and perceived-reality discussion."),
    (13, 8, "clear", "Distinct solar, wind, panels, grids, batteries, electricity, and renewable-cost discussion."),
    (13, 9, "clear", "Distinct scientific attribution and consensus discussion about human-caused warming."),
    (13, 10, "duplicate", "General fossil-fuel use and renewable-transition language overlaps adjacent fossil-energy clusters."),
    (13, 11, "duplicate", "Fossil-fuel industry and company language duplicates the broader fossil-transition cluster."),
    (13, 12, "duplicate", "Oil and gas company, price, production, and political-economy language overlaps the other fossil clusters."),
]

baseline_cluster_review = pd.DataFrame(
    baseline_cluster_review_rows,
    columns=["k", "cluster", "review_status", "review_rationale"],
)
baseline_cluster_review.insert(0, "method", "tfidf_lsa_kmeans")
baseline_cluster_review = baseline_cluster_review.merge(
    extended_candidate_profiles[[
        "method", "k", "cluster", "size", "share", "mean_membership", "top_terms"
    ]],
    on=["method", "k", "cluster"],
    how="left",
    validate="one_to_one",
)
assert baseline_cluster_review["top_terms"].notna().all()
baseline_review_csv_path = write_review_csv(
    baseline_cluster_review,
    "tfidf_lsa_candidate_cluster_review.csv",
)
display(baseline_cluster_review)

all_candidate_cluster_review = pd.concat(
    [minilm_cluster_review, baseline_cluster_review],
    ignore_index=True,
)
all_review_csv_path = write_review_csv(
    all_candidate_cluster_review,
    "all_candidate_cluster_review.csv",
)

baseline_review_summary = (
    pd.crosstab(
        [baseline_cluster_review["method"], baseline_cluster_review["k"]],
        baseline_cluster_review["review_status"],
    )
    .reindex(columns=status_order, fill_value=0)
    .reset_index()
    .rename(columns={status: status + "_clusters" for status in status_order})
)
baseline_notes = pd.DataFrame([
    (5, "Very stable but too coarse; one large mixed cluster combines several substantively different lexical themes."),
    (11, "Useful lexical coverage with one overlapping fossil-transition cluster and less fragmentation than k=12 or k=13."),
    (12, "Additional carbon and fossil-energy splits create two overlaps without materially improving the sensitivity role."),
    (13, "Higher resolution, but three overlapping fossil-fuel, fuel-industry, and oil-and-gas clusters reduce parsimony."),
], columns=["k", "review_note"])
baseline_review_summary = baseline_review_summary.merge(
    baseline_notes,
    on="k",
    validate="one_to_one",
)

review_summary = pd.concat([minilm_review_summary, baseline_review_summary], ignore_index=True)
review_comparison = review_summary.merge(
    selection_scores[[
        "method", "k", "passes_guardrails", "quantitative_rank",
        "quantitative_score", "silhouette_native", "mean_seed_ari",
        "minimum_cluster_share", "maximum_cluster_share",
    ]],
    on=["method", "k"],
    how="left",
    validate="one_to_one",
)

review_comparison["interpretability_score"] = (
    review_comparison["clear_clusters"]
    + 0.5 * review_comparison["broad_clusters"]
    - review_comparison["mixed_clusters"]
    - review_comparison["duplicate_clusters"]
    - review_comparison["residual_clusters"]
) / review_comparison["k"]
review_comparison["structural_issue_count"] = (
    review_comparison["mixed_clusters"]
    + review_comparison["duplicate_clusters"]
    + review_comparison["residual_clusters"]
)
review_comparison["interpretability_rank"] = (
    review_comparison.groupby("method")["interpretability_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)
review_comparison["calculated_selection"] = False
for method, group in review_comparison.loc[review_comparison["passes_guardrails"]].groupby("method"):
    winner_index = group.sort_values(
        ["interpretability_score", "quantitative_score"],
        ascending=[False, False],
    ).index[0]
    review_comparison.loc[winner_index, "calculated_selection"] = True

review_comparison.to_csv(
    OUTPUT_DIR / "comparable_interpretation_review_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

review_columns = [
    "method", "k", "passes_guardrails", "quantitative_rank", "quantitative_score",
    "silhouette_native", "mean_seed_ari", "minimum_cluster_share", "maximum_cluster_share",
    "clear_clusters", "broad_clusters", "mixed_clusters", "duplicate_clusters",
    "residual_clusters", "structural_issue_count", "interpretability_score",
    "interpretability_rank", "calculated_selection", "review_note",
]
display(
    review_comparison[review_columns]
    .sort_values(["method", "interpretability_rank"])
    .round(4)
)


Saved: D:\DM1_project\going to submit\09e_outputs\minilm_candidate_cluster_review.csv


,method,k,cluster,review_status,review_rationale,size,share,mean_membership,top_terms
0,minilm_kmeans,4,0,clear,"Distinct EV, battery, and charging discussion.",1978,0.070067,0.631960,"ev, car, electric, battery, vehicles, batterie..."
1,minilm_kmeans,4,1,broad,"Combines temperature, weather, ice, data, and ...",9635,0.341304,0.603976,"warming, co2, earth, temperature, weather, ice..."
2,minilm_kmeans,4,2,broad,"Combines emissions, carbon, fossil fuels, and ...",10005,0.354410,0.556802,"carbon, emissions, oil, fossil, co2, energy, g..."
3,minilm_kmeans,4,3,mixed,"Nuclear, renewables, grids, coal, and system-c...",6612,0.234219,0.615468,"energy, solar, power, nuclear, wind, renewable..."
4,minilm_kmeans,6,0,mixed,Nuclear and renewable-grid technologies remain...,5729,0.202940,0.626398,"solar, energy, power, nuclear, wind, grid, ren..."
5,minilm_kmeans,6,1,clear,Distinct fossil-fuel industry and transition p...,4809,0.170351,0.534302,"oil, energy, trump, fossil, gas, companies, mo..."
6,minilm_kmeans,6,2,broad,"Climate belief, future risk, and human-consequ...",6666,0.236132,0.650406,"warming, planet, problem, humans, real, life, ..."
7,minilm_kmeans,6,3,clear,"Distinct EV, battery, and charging discussion.",1907,0.067552,0.637232,"ev, car, electric, battery, vehicles, ice, bat..."
8,minilm_kmeans,6,4,clear,"Distinct climate observation, weather, ice, an...",4393,0.155615,0.583816,"warming, temperature, ice, weather, earth, dat..."
9,minilm_kmeans,6,5,broad,"Combines greenhouse gases, emissions responsib...",4726,0.167411,0.616999,"carbon, co2, emissions, fossil, energy, use, a..."


Saved: D:\DM1_project\going to submit\09e_outputs\tfidf_lsa_candidate_cluster_review.csv


,method,k,cluster,review_status,review_rationale,size,share,mean_membership,top_terms
0,tfidf_lsa_kmeans,5,0,clear,Coherent physical-climate observations and imp...,5385,0.190755,0.402495,"co2, warming, temperature, earth, ice, water, ..."
1,tfidf_lsa_kmeans,5,1,broad,"Broad transition theme combining fossil fuels,...",2166,0.076727,0.549043,"fossil, fuels, energy, renewables, industry, u..."
2,tfidf_lsa_kmeans,5,2,clear,Coherent public-belief and science-trust discu...,5183,0.183599,0.555385,"real, believe, humans, problem, science, point..."
3,tfidf_lsa_kmeans,5,3,broad,Multiple electricity technologies remain combi...,5858,0.207510,0.414914,"solar, energy, power, nuclear, wind, grid, coa..."
4,tfidf_lsa_kmeans,5,4,mixed,"Oil, emissions, EVs, China, corporate interest...",9638,0.341410,0.360729,"oil, carbon, emissions, gas, ev, energy, china..."
5,tfidf_lsa_kmeans,11,0,broad,Broad electricity-system theme combining nucle...,3691,0.130747,0.413292,"energy, nuclear, power, coal, plants, renewabl..."
6,tfidf_lsa_kmeans,11,1,clear,"Distinct public belief, science trust, and per...",4016,0.142260,0.586441,"real, believe, problem, humans, science, point..."
7,tfidf_lsa_kmeans,11,2,broad,"Broad human-environment theme combining food, ...",5727,0.202869,0.400750,"carbon, planet, environment, good, life, food,..."
8,tfidf_lsa_kmeans,11,3,broad,"Physical-impact theme spans weather, temperatu...",2822,0.099965,0.429518,"ice, weather, temperature, sea, water, heat, d..."
9,tfidf_lsa_kmeans,11,4,clear,"Distinct oil and gas industry, prices, product...",1706,0.060432,0.535535,"oil, gas, companies, energy, big, industry, pr..."


Saved: D:\DM1_project\going to submit\09e_outputs\all_candidate_cluster_review.csv


,method,k,passes_guardrails,quantitative_rank,quantitative_score,silhouette_native,mean_seed_ari,minimum_cluster_share,maximum_cluster_share,clear_clusters,broad_clusters,mixed_clusters,duplicate_clusters,residual_clusters,structural_issue_count,interpretability_score,interpretability_rank,calculated_selection,review_note
3,minilm_kmeans,9,True,4,0.6699,0.0705,0.9752,0.0482,0.1559,6,3,0,0,0,0,0.8333,1,True,"Nine substantive themes with no mixed, duplica..."
2,minilm_kmeans,7,True,2,0.7462,0.0648,0.9871,0.0668,0.1882,4,3,0,0,0,0,0.7857,2,False,"Clearer, but nuclear and renewable-grid discus..."
1,minilm_kmeans,6,True,1,0.7866,0.0680,0.9944,0.0676,0.2361,3,2,1,0,0,1,0.5000,3,False,Stable but too coarse: nuclear/renewables and ...
0,minilm_kmeans,4,True,3,0.7408,0.0822,0.8272,0.0701,0.3544,1,2,1,0,0,1,0.2500,4,False,"Too coarse: physical climate, carbon/fossil, a..."
5,tfidf_lsa_kmeans,11,True,3,0.5148,0.0984,0.8015,0.0412,0.2029,6,4,0,1,0,1,0.6364,1,True,Useful lexical coverage with one overlapping f...
6,tfidf_lsa_kmeans,12,True,4,0.5115,0.1009,0.8161,0.0307,0.2022,7,3,0,2,0,2,0.5417,2,False,Additional carbon and fossil-energy splits cre...
7,tfidf_lsa_kmeans,13,True,2,0.5343,0.1044,0.8463,0.0312,0.1802,8,2,0,3,0,3,0.4615,3,False,"Higher resolution, but three overlapping fossi..."
4,tfidf_lsa_kmeans,5,True,1,0.6867,0.0715,0.9969,0.0767,0.3414,2,2,1,0,0,1,0.4000,4,False,Very stable but too coarse; one large mixed cl...


## 9. Final choice and exports / 最终选择与导出

The selected row for each method is used to rebuild assignments and save the inputs needed by 09F.

每种方法按上方计算结果选出一个方案，重建归属并保存 09F 所需的数据。

In [10]:
advanced_choice = review_comparison.loc[
    review_comparison["method"].eq("minilm_kmeans")
    & review_comparison["calculated_selection"]
].iloc[0]
baseline_choice = review_comparison.loc[
    review_comparison["method"].eq("tfidf_lsa_kmeans")
    & review_comparison["calculated_selection"]
].iloc[0]

ADVANCED_METHOD = str(advanced_choice["method"])
ADVANCED_K = int(advanced_choice["k"])
BASELINE_METHOD = str(baseline_choice["method"])
BASELINE_K = int(baseline_choice["k"])
PRIMARY_METHOD = ADVANCED_METHOD
PRIMARY_K = ADVANCED_K

print("Calculated primary choice:", ADVANCED_METHOD, ADVANCED_K)
print("Calculated baseline choice:", BASELINE_METHOD, BASELINE_K)

baseline_model = all_models[(BASELINE_METHOD, BASELINE_K)]
baseline_labels = all_labels[(BASELINE_METHOD, BASELINE_K)]
baseline_strength, baseline_margin = normalized_centroid_membership(
    X_lsa, baseline_labels, baseline_model.cluster_centers_
)

advanced_model = all_models[(ADVANCED_METHOD, ADVANCED_K)]
advanced_labels = all_labels[(ADVANCED_METHOD, ADVANCED_K)]
advanced_strength, advanced_margin = normalized_centroid_membership(
    X_semantic, advanced_labels, advanced_model.cluster_centers_
)

selected = df.copy()
selected["baseline_selected_k"] = BASELINE_K
selected["baseline_selected_cluster"] = baseline_labels
selected["baseline_selected_membership_strength"] = baseline_strength
selected["baseline_selected_margin"] = baseline_margin
selected["advanced_selected_k"] = ADVANCED_K
selected["advanced_selected_cluster"] = advanced_labels
selected["advanced_selected_membership_strength"] = advanced_strength
selected["advanced_selected_margin"] = advanced_margin
selected["primary_method"] = PRIMARY_METHOD
selected["primary_k"] = PRIMARY_K
selected["primary_cluster"] = advanced_labels
selected["primary_membership_strength"] = advanced_strength
selected["primary_margin"] = advanced_margin
selected.to_parquet(OUTPUT_DIR / "09e_selected_assignments.parquet", index=False)

joblib.dump(
    baseline_model,
    OUTPUT_DIR / f"selected_{BASELINE_METHOD}_k{BASELINE_K}.joblib",
)
joblib.dump(
    advanced_model,
    OUTPUT_DIR / f"selected_{ADVANCED_METHOD}_k{ADVANCED_K}.joblib",
)


def metric_record(method, k):
    row = selection_scores.loc[
        selection_scores["method"].eq(method) & selection_scores["k"].eq(k)
    ].iloc[0]
    keys = [
        "quantitative_rank", "quantitative_score", "silhouette_lsa",
        "silhouette_minilm", "mean_seed_ari", "minimum_seed_ari",
        "minimum_cluster_share", "maximum_cluster_share",
        "normalized_size_entropy", "topic_diversity",
        "mean_term_exclusivity", "mean_native_margin",
    ]
    return {key: float(row[key]) for key in keys}


score_only_winners = (
    selection_scores.loc[selection_scores["passes_guardrails"]]
    .sort_values(["method", "quantitative_score"], ascending=[True, False])
    .groupby("method", as_index=False, group_keys=False)
    .head(1)
    .set_index("method")["k"]
    .astype(int)
    .to_dict()
)

decision = {
    "status": "final_after_comparable_interpretation_review",
    "n_comments": int(len(df)),
    "data_and_representation": "exact_09c_population_and_weighted_representations",
    "score_only_winners": {
        "baseline": {
            "method": "tfidf_lsa_kmeans",
            "k": score_only_winners["tfidf_lsa_kmeans"],
        },
        "advanced": {
            "method": "minilm_kmeans",
            "k": score_only_winners["minilm_kmeans"],
        },
    },
    "baseline_selection": {
        "method": BASELINE_METHOD,
        "k": BASELINE_K,
        "metrics": metric_record(BASELINE_METHOD, BASELINE_K),
        "role": "lexical_sensitivity_baseline",
        "reason": str(baseline_choice["review_note"]),
        "interpretability_score": float(baseline_choice["interpretability_score"]),
    },
    "advanced_selection": {
        "method": ADVANCED_METHOD,
        "k": ADVANCED_K,
        "metrics": metric_record(ADVANCED_METHOD, ADVANCED_K),
        "role": "primary_topic_model",
        "reason": str(advanced_choice["review_note"]),
        "interpretability_score": float(advanced_choice["interpretability_score"]),
    },
    "primary_selection": {"method": PRIMARY_METHOD, "k": PRIMARY_K},
    "review_type": "cluster-level semantic review; not an independent gold standard",
    "review_policy": "same centroid/random sampling rule across candidates; clarity, breadth, mixing, duplication, and residual structure reviewed with quantitative diagnostics",
    "important_limit": "The final choice is a multi-criterion analytical decision, not a claim that one scalar metric identifies a uniquely true k.",
}

(OUTPUT_DIR / "final_selection_decision.json").write_text(
    json.dumps(decision, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

final_selection_table = pd.DataFrame([
    {
        "role": "Primary semantic model",
        "method": ADVANCED_METHOD,
        "selected_k": ADVANCED_K,
        "score_only_rank": int(decision["advanced_selection"]["metrics"]["quantitative_rank"]),
        "native_silhouette": decision["advanced_selection"]["metrics"]["silhouette_minilm"],
        "mean_seed_ARI": decision["advanced_selection"]["metrics"]["mean_seed_ari"],
        "cluster_share_range": f"{decision['advanced_selection']['metrics']['minimum_cluster_share']:.1%}–{decision['advanced_selection']['metrics']['maximum_cluster_share']:.1%}",
        "final_decision": "SELECTED",
    },
    {
        "role": "Lexical sensitivity baseline",
        "method": BASELINE_METHOD,
        "selected_k": BASELINE_K,
        "score_only_rank": int(decision["baseline_selection"]["metrics"]["quantitative_rank"]),
        "native_silhouette": decision["baseline_selection"]["metrics"]["silhouette_lsa"],
        "mean_seed_ARI": decision["baseline_selection"]["metrics"]["mean_seed_ari"],
        "cluster_share_range": f"{decision['baseline_selection']['metrics']['minimum_cluster_share']:.1%}–{decision['baseline_selection']['metrics']['maximum_cluster_share']:.1%}",
        "final_decision": "SELECTED AS BASELINE",
    },
])

display(final_selection_table.round(4))
print("FINAL PRIMARY MODEL:", PRIMARY_METHOD, f"k={PRIMARY_K}")
print("FINAL LEXICAL BASELINE:", BASELINE_METHOD, f"k={BASELINE_K}")
print("Saved final decision:", OUTPUT_DIR / "final_selection_decision.json")
print("Saved selected assignments:", OUTPUT_DIR / "09e_selected_assignments.parquet")


Calculated primary choice: minilm_kmeans 9
Calculated baseline choice: tfidf_lsa_kmeans 11


,role,method,selected_k,score_only_rank,native_silhouette,mean_seed_ARI,cluster_share_range,final_decision
0,Primary semantic model,minilm_kmeans,9,4,0.0705,0.9752,4.8%–15.6%,SELECTED
1,Lexical sensitivity baseline,tfidf_lsa_kmeans,11,3,0.0984,0.8015,4.1%–20.3%,SELECTED AS BASELINE


FINAL PRIMARY MODEL: minilm_kmeans k=9
FINAL LEXICAL BASELINE: tfidf_lsa_kmeans k=11
Saved final decision: D:\DM1_project\going to submit\09e_outputs\final_selection_decision.json
Saved selected assignments: D:\DM1_project\going to submit\09e_outputs\09e_selected_assignments.parquet


In [11]:
primary_metrics = decision["advanced_selection"]["metrics"]
baseline_metrics = decision["baseline_selection"]["metrics"]

conclusion = f"""# 09E final model-selection conclusion / 09E 最终模型选择结论

## 中文

09E 不继承旧的 k=12。它在相同的 09c 样本和表示上独立评价 k=4–20。MiniLM k=6 获得最高量化综合分，但统一解释性复核发现其仍合并核能与可再生能源，以及气候信念与未来风险。MiniLM k=9 通过全部量化门槛，并形成九个实质性、均衡且不重复的主题，因此被选为主要分析模型。

最终主要模型为 weighted MiniLM KMeans, k={ADVANCED_K}。其 MiniLM silhouette 为 {primary_metrics['silhouette_minilm']:.4f}，五随机种子平均 ARI 为 {primary_metrics['mean_seed_ari']:.3f}，簇占比范围为 {primary_metrics['minimum_cluster_share']:.1%}–{primary_metrics['maximum_cluster_share']:.1%}。

最终词汇敏感性基线为 TF-IDF + LSA KMeans, k={BASELINE_K}。其 LSA silhouette 为 {baseline_metrics['silhouette_lsa']:.4f}，五随机种子平均 ARI 为 {baseline_metrics['mean_seed_ari']:.3f}。基线仅用于检验结果对词汇表示的敏感性。

## English

Notebook 09E evaluates k=4–20 on the same 09c population and representations. MiniLM k=6 achieves the highest quantitative score, but the common interpretation review shows that it still merges nuclear with renewable energy and climate belief with future risk. MiniLM k=9 passes every quantitative guardrail and provides nine substantive, balanced, and nonredundant themes, so it is selected as the primary analytical model.

The final primary model is weighted MiniLM KMeans with k={ADVANCED_K}. The lexical sensitivity baseline is TF-IDF + LSA KMeans with k={BASELINE_K}. The choice is a multi-criterion analytical decision rather than the result of one scalar metric.
"""

(OUTPUT_DIR / "09e_model_selection_conclusion.md").write_text(
    conclusion,
    encoding="utf-8",
)
print(conclusion)


# 09E final model-selection conclusion / 09E 最终模型选择结论

## 中文

09E 不继承旧的 k=12。它在相同的 09c 样本和表示上独立评价 k=4–20。MiniLM k=6 获得最高量化综合分，但统一解释性复核发现其仍合并核能与可再生能源，以及气候信念与未来风险。MiniLM k=9 通过全部量化门槛，并形成九个实质性、均衡且不重复的主题，因此被选为主要分析模型。

最终主要模型为 weighted MiniLM KMeans, k=9。其 MiniLM silhouette 为 0.0705，五随机种子平均 ARI 为 0.975，簇占比范围为 4.8%–15.6%。

最终词汇敏感性基线为 TF-IDF + LSA KMeans, k=11。其 LSA silhouette 为 0.0984，五随机种子平均 ARI 为 0.801。基线仅用于检验结果对词汇表示的敏感性。

## English

Notebook 09E evaluates k=4–20 on the same 09c population and representations. MiniLM k=6 achieves the highest quantitative score, but the common interpretation review shows that it still merges nuclear with renewable energy and climate belief with future risk. MiniLM k=9 passes every quantitative guardrail and provides nine substantive, balanced, and nonredundant themes, so it is selected as the primary analytical model.

The final primary model is weighted MiniLM KMeans with k=9. The lexical sensitivity baseline is TF-IDF + LSA KMeans with k=11. The choice i

## Run order / 运行顺序

Run 09E from top to bottom. It saves `09e_selected_assignments.parquet`, `final_selection_decision.json`, the review tables, and the MiniLM vectors used by 09F.

从头到尾运行 09E 后，会生成 `09e_selected_assignments.parquet`、`final_selection_decision.json`、复核表和 09F 使用的 MiniLM 向量。